# vLLM JSON Schema Workaround - Interactive Demo

This notebook demonstrates how to get structured JSON output from vLLM models that **don't support** `response_format` or tool calling.

**What you'll learn:**
1. How to manually inject JSON schema into prompts
2. How to validate the prompt transformation (without a real endpoint)
3. How to use LiteLLM's logging to see what gets sent to the model
4. Best practices for getting reliable JSON output

**Key Point:** LiteLLM does NOT automatically inject JSON schema for vLLM. You must do it manually.

## Setup

First, let's install dependencies and set up logging to see what's happening.

In [ ]:
# Install if needed (uncomment if running in a fresh environment)
# !pip install litellm pydantic

import litellm
import json
import logging
from pydantic import BaseModel, Field
from typing import List, Optional

# Enable detailed logging to see what LiteLLM sends
litellm.set_verbose = True

# Set up logging to capture what's sent to the model
logging.basicConfig(level=logging.DEBUG)
logger = logging.getLogger('litellm')

print("✅ Setup complete!")

## Part 1: Demonstrate the Problem

Let's show what happens when you try to use `response_format` with vLLM (it just passes through).

In [ ]:
# Create a custom callback to intercept the request
class RequestInspector:
    def __init__(self):
        self.last_request = None
        self.last_messages = None
    
    def log_pre_api_call(self, model, messages, kwargs):
        """Called before the API request is made"""
        self.last_messages = messages
        self.last_request = kwargs
        print("\n" + "="*80)
        print("📤 INTERCEPTED REQUEST TO MODEL")
        print("="*80)
        print(f"\n🎯 Model: {model}")
        print(f"\n💬 Messages being sent ({len(messages)} total):")
        for i, msg in enumerate(messages, 1):
            role = msg.get('role', 'unknown')
            content = msg.get('content', '')
            # Truncate long content for display
            display_content = content if len(str(content)) < 200 else str(content)[:200] + "..."
            print(f"\n  [{i}] {role.upper()}:")
            print(f"      {display_content}")
        
        print(f"\n⚙️  Optional Parameters:")
        for key, value in kwargs.items():
            if key not in ['messages', 'model']:
                print(f"      {key}: {value}")
        print("\n" + "="*80 + "\n")

# Create inspector
inspector = RequestInspector()

# Register callback
litellm.callbacks = [inspector.log_pre_api_call]

print("✅ Request inspector ready!")

In [ ]:
# Define a schema we want the model to follow
schema_dict = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "age": {"type": "integer"},
        "email": {"type": "string"},
        "occupation": {"type": "string"}
    },
    "required": ["name", "age", "email"]
}

messages = [
    {"role": "user", "content": "Extract user info: John Doe, 30 years old, john@example.com, Software Engineer"}
]

print("🔴 ATTEMPT 1: Using response_format (WRONG - won't inject schema)\n")

try:
    # This will show that response_format just passes through
    # We'll use a mock API to prevent actual call
    litellm.drop_params = True  # Don't error on unsupported params
    
    # Mock the actual HTTP call so we can see the request without needing an endpoint
    from unittest.mock import patch, MagicMock
    
    with patch('litellm.llms.custom_httpx.http_handler.HTTPHandler.post') as mock_post:
        # Make it raise an exception so we don't actually call anything
        mock_post.side_effect = Exception("MOCK: Not making real API call")
        
        try:
            response = litellm.completion(
                model="hosted_vllm/my-model",
                messages=messages,
                response_format={
                    "type": "json_schema",
                    "json_schema": {
                        "name": "user_extraction",
                        "schema": schema_dict
                    }
                },
                api_base="http://mock-vllm-server:8000"
            )
        except Exception as e:
            if "MOCK" in str(e):
                print("\n✅ Request captured (mock API call prevented)")
            else:
                raise

except Exception as e:
    print(f"\n⚠️  Error (expected): {e}")

print("\n🔍 OBSERVATION:")
print("   Notice that the messages array has ONLY 1 message.")
print("   The JSON schema was NOT injected into the messages!")
print("   It's just passed in response_format parameter (which vLLM might ignore).")

## Part 2: The Correct Approach - Manual Injection

Now let's manually add the JSON schema to the messages.

In [ ]:
def create_json_schema_message(schema: dict, verbose: bool = True) -> str:
    """
    Creates a message that instructs the model to follow a JSON schema.
    This mimics what LiteLLM does for Gemini.
    """
    message = f"""You must respond with valid JSON that matches this exact schema:

```json
{json.dumps(schema, indent=2)}
```

IMPORTANT RULES:
- Output ONLY the JSON object (start with {{ and end with }})
- Do NOT include markdown code blocks (no ```json)
- Do NOT add any explanations before or after the JSON
- All required fields must be present
- Use the exact property names from the schema"""
    
    if verbose:
        print("📝 Generated schema instruction message:")
        print("─" * 80)
        print(message)
        print("─" * 80)
    
    return message

# Create the schema instruction
schema_message = create_json_schema_message(schema_dict)

In [ ]:
# Now create messages WITH the schema manually injected
messages_with_schema = [
    {
        "role": "system",
        "content": "You are a helpful assistant that outputs valid JSON only."
    },
    {
        "role": "user",
        "content": "Extract user info: John Doe, 30 years old, john@example.com, Software Engineer"
    },
    {
        "role": "user",
        "content": schema_message
    }
]

print("\n🟢 ATTEMPT 2: Manual schema injection (CORRECT)\n")

try:
    from unittest.mock import patch
    
    with patch('litellm.llms.custom_httpx.http_handler.HTTPHandler.post') as mock_post:
        mock_post.side_effect = Exception("MOCK: Not making real API call")
        
        try:
            response = litellm.completion(
                model="hosted_vllm/my-model",
                messages=messages_with_schema,
                api_base="http://mock-vllm-server:8000",
                temperature=0  # Low temperature for more deterministic output
            )
        except Exception as e:
            if "MOCK" in str(e):
                print("\n✅ Request captured (mock API call prevented)")
            else:
                raise

except Exception as e:
    print(f"\n⚠️  Error: {e}")

print("\n🔍 OBSERVATION:")
print("   Notice that the messages array now has 3 messages:")
print("   1. System message (JSON-only assistant)")
print("   2. User query (extract user info)")
print("   3. User message with JSON schema instructions")
print("\n   ✅ The schema IS in the prompt that will be sent to the model!")

## Part 3: Reusable Helper Function

Let's create a helper that automatically does this for us.

In [ ]:
def completion_with_schema(
    model: str,
    messages: list,
    schema: dict,
    api_base: Optional[str] = None,
    show_transformed_messages: bool = True,
    **kwargs
):
    """
    Workaround for vLLM models without native response_format support.
    Manually injects JSON schema into the messages.
    
    Args:
        model: Model name (e.g., "hosted_vllm/llama-3-8b")
        messages: List of message dicts
        schema: JSON schema dict or Pydantic model
        api_base: vLLM server URL
        show_transformed_messages: If True, prints the messages before sending
        **kwargs: Additional params for litellm.completion
    
    Returns:
        Parsed JSON dict (or raises ValueError if invalid JSON)
    """
    # Convert Pydantic to dict if needed
    if hasattr(schema, 'model_json_schema'):
        schema_dict = schema.model_json_schema()
        pydantic_class = schema
    else:
        schema_dict = schema
        pydantic_class = None
    
    # Create schema instruction message
    schema_instruction = create_json_schema_message(schema_dict, verbose=False)
    
    # Add schema to messages
    messages_with_schema = messages + [
        {"role": "user", "content": schema_instruction}
    ]
    
    if show_transformed_messages:
        print("\n📨 TRANSFORMED MESSAGES (what will be sent to model):")
        print("="*80)
        for i, msg in enumerate(messages_with_schema, 1):
            print(f"\n[{i}] {msg['role'].upper()}:")
            content = str(msg['content'])
            if len(content) > 150:
                print(f"  {content[:150]}...")
            else:
                print(f"  {content}")
        print("\n" + "="*80 + "\n")
    
    # Call LiteLLM
    litellm.drop_params = True
    response = litellm.completion(
        model=model,
        api_base=api_base,
        messages=messages_with_schema,
        temperature=kwargs.pop('temperature', 0),
        **kwargs
    )
    
    # Parse response (with markdown unwrapping)
    content = response.choices[0].message.content
    
    try:
        result = json.loads(content)
    except json.JSONDecodeError:
        # Try to extract from markdown code block
        import re
        match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', content, re.DOTALL)
        if match:
            result = json.loads(match.group(1))
        else:
            # Try to find any JSON object
            match = re.search(r'\{.*\}', content, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
            else:
                raise ValueError(f"No valid JSON found in response: {content}")
    
    # Validate with Pydantic if provided
    if pydantic_class:
        return pydantic_class.model_validate(result)
    
    return result

print("✅ Helper function defined!")

In [ ]:
# Test the helper function
print("🧪 Testing helper function with Pydantic model\n")

class UserInfo(BaseModel):
    name: str = Field(description="Full name")
    age: int = Field(ge=0, le=150, description="Age in years")
    email: str = Field(description="Email address")
    occupation: Optional[str] = Field(default=None, description="Job title")

messages = [
    {"role": "user", "content": "Extract: Jane Smith, 25, jane@example.com, Data Scientist"}
]

try:
    from unittest.mock import patch
    
    with patch('litellm.llms.custom_httpx.http_handler.HTTPHandler.post') as mock_post:
        mock_post.side_effect = Exception("MOCK: Not making real API call")
        
        try:
            result = completion_with_schema(
                model="hosted_vllm/my-model",
                messages=messages,
                schema=UserInfo,
                api_base="http://mock-vllm-server:8000",
                show_transformed_messages=True
            )
        except Exception as e:
            if "MOCK" in str(e):
                print("\n✅ Request successfully transformed and captured!")
                print("\nIn a real scenario with a working vLLM endpoint:")
                print("  - These messages would be sent to the model")
                print("  - The model would see the schema in the prompt")
                print("  - It would (hopefully) respond with valid JSON")
            else:
                raise
                
except Exception as e:
    print(f"Error: {e}")

## Part 4: Comparing Different Template Styles

Let's test different ways to phrase the schema instruction to see what gets sent.

In [ ]:
# Template 1: Concise
def template_concise(schema: dict) -> str:
    return f"""Respond with JSON matching this schema:\n{json.dumps(schema, indent=2)}"""

# Template 2: Explicit (what we used above)
def template_explicit(schema: dict) -> str:
    return f"""You must respond with valid JSON that matches this exact schema:

```json
{json.dumps(schema, indent=2)}
```

IMPORTANT RULES:
- Output ONLY the JSON object
- Do NOT include markdown code blocks
- Do NOT add explanations
- All required fields must be present"""

# Template 3: Instruction-tuned model style
def template_instruction_tuned(schema: dict) -> str:
    return f"""### Task
Extract information and format as JSON.

### Output Schema
```json
{json.dumps(schema, indent=2)}
```

### Requirements
1. Output valid JSON only (no markdown)
2. Match the schema exactly
3. Include all required fields

### Response
"""

# Template 4: Claude-style
def template_claude_style(schema: dict) -> str:
    return f"""<instructions>
You must output a valid JSON object that conforms to this schema:

<schema>
{json.dumps(schema, indent=2)}
</schema>

Output only the JSON object. Do not include any explanations or markdown formatting.
</instructions>"""

# Test each template
templates = {
    "Concise": template_concise,
    "Explicit": template_explicit,
    "Instruction-tuned": template_instruction_tuned,
    "Claude-style": template_claude_style
}

simple_schema = {
    "type": "object",
    "properties": {
        "color": {"type": "string"},
        "count": {"type": "integer"}
    },
    "required": ["color", "count"]
}

print("📊 COMPARING TEMPLATE STYLES\n")
print("="*80)

for name, template_func in templates.items():
    print(f"\n🎨 {name.upper()} TEMPLATE:")
    print("─"*80)
    message = template_func(simple_schema)
    print(message)
    print("─"*80)
    print(f"Character count: {len(message)}")
    print()

## Part 5: Validation - Inspect Final Request

Let's create a comprehensive test to validate everything is working correctly.

In [ ]:
def validate_schema_injection(messages: list, schema: dict, template_func=template_explicit):
    """
    Validates that schema injection is working correctly.
    Returns a report of what will be sent to the model.
    """
    print("\n" + "="*80)
    print("🔬 SCHEMA INJECTION VALIDATION")
    print("="*80)
    
    # Original messages
    print(f"\n📥 ORIGINAL MESSAGES ({len(messages)} total):")
    for i, msg in enumerate(messages, 1):
        print(f"  [{i}] {msg['role']}: {msg['content'][:60]}...")
    
    # Generate schema instruction
    schema_instruction = template_func(schema)
    
    # Create final messages
    final_messages = messages + [{"role": "user", "content": schema_instruction}]
    
    print(f"\n📤 FINAL MESSAGES ({len(final_messages)} total):")
    for i, msg in enumerate(final_messages, 1):
        content = msg['content']
        preview = content[:60] + "..." if len(content) > 60 else content
        print(f"  [{i}] {msg['role']}: {preview}")
    
    # Validation checks
    print("\n✅ VALIDATION CHECKS:")
    
    # Check 1: Schema is present
    schema_str = json.dumps(schema)
    schema_in_messages = any(schema_str in str(msg.get('content', '')) for msg in final_messages)
    print(f"  {'✅' if schema_in_messages else '❌'} Schema is present in messages")
    
    # Check 2: Message count increased
    count_increased = len(final_messages) > len(messages)
    print(f"  {'✅' if count_increased else '❌'} Message count increased ({len(messages)} → {len(final_messages)})")
    
    # Check 3: Contains JSON formatting instructions
    has_instructions = any('JSON' in str(msg.get('content', '')).upper() for msg in final_messages)
    print(f"  {'✅' if has_instructions else '❌'} Contains JSON formatting instructions")
    
    # Check 4: Schema properties are mentioned
    properties_mentioned = all(
        prop in str(final_messages)
        for prop in schema.get('properties', {}).keys()
    )
    print(f"  {'✅' if properties_mentioned else '❌'} All schema properties are mentioned")
    
    # Full message dump
    print("\n📄 COMPLETE FINAL MESSAGES:")
    print("="*80)
    print(json.dumps(final_messages, indent=2))
    print("="*80)
    
    # Summary
    all_checks_passed = schema_in_messages and count_increased and has_instructions and properties_mentioned
    print(f"\n{'🎉 ALL CHECKS PASSED!' if all_checks_passed else '⚠️  Some checks failed'}")
    
    return final_messages

# Run validation
test_messages = [
    {"role": "system", "content": "You are a helpful JSON-only assistant."},
    {"role": "user", "content": "What are 3 primary colors and how many are there?"}
]

test_schema = {
    "type": "object",
    "properties": {
        "colors": {
            "type": "array",
            "items": {"type": "string"}
        },
        "count": {"type": "integer"}
    },
    "required": ["colors", "count"]
}

final = validate_schema_injection(test_messages, test_schema)

## Part 6: Custom Template System (Like LiteLLM's)

Let's implement a custom prompt template system similar to LiteLLM's `custom_prompt_dict`.

In [ ]:
# Custom template registry
CUSTOM_TEMPLATES = {}

def register_response_schema_template(model_name: str, template_config: dict):
    """
    Register a custom template for a specific model.
    
    template_config should have:
    - 'pre_message': Text before the schema
    - 'post_message': Text after the schema
    - 'schema_format': 'json' or 'yaml' or 'inline'
    """
    CUSTOM_TEMPLATES[model_name] = template_config
    print(f"✅ Registered template for {model_name}")

def get_schema_message(model_name: str, schema: dict) -> str:
    """
    Get the schema message using custom template if registered,
    otherwise use default.
    """
    if model_name in CUSTOM_TEMPLATES:
        config = CUSTOM_TEMPLATES[model_name]
        pre = config.get('pre_message', '')
        post = config.get('post_message', '')
        schema_format = config.get('schema_format', 'json')
        
        if schema_format == 'json':
            schema_str = f"```json\n{json.dumps(schema, indent=2)}\n```"
        elif schema_format == 'inline':
            schema_str = json.dumps(schema)
        else:
            schema_str = json.dumps(schema, indent=2)
        
        return f"{pre}\n{schema_str}\n{post}"
    else:
        # Default template
        return template_explicit(schema)

# Register different templates for different models
register_response_schema_template(
    "hosted_vllm/llama-3-8b",
    {
        'pre_message': '### Instructions\nRespond with JSON matching this schema:',
        'post_message': '\n### Rules\n- Output valid JSON only\n- No explanations\n- Match schema exactly',
        'schema_format': 'json'
    }
)

register_response_schema_template(
    "hosted_vllm/qwen-7b",
    {
        'pre_message': '<|im_start|>system\nOutput JSON with this structure:',
        'post_message': '<|im_end|>',
        'schema_format': 'inline'
    }
)

# Test different templates
print("\n📝 TESTING CUSTOM TEMPLATES\n")

test_schema = {"type": "object", "properties": {"result": {"type": "string"}}}

for model_name in ["hosted_vllm/llama-3-8b", "hosted_vllm/qwen-7b", "hosted_vllm/unknown-model"]:
    print(f"\n🎯 {model_name}:")
    print("─"*80)
    message = get_schema_message(model_name, test_schema)
    print(message)
    print("─"*80)

## Summary

### What We Learned:

1. **LiteLLM does NOT automatically inject JSON schema for vLLM**
   - `response_format` just passes through to the vLLM server
   - If your model doesn't support it natively, it gets ignored

2. **Manual injection is required**
   - Add a user message with the JSON schema and instructions
   - Different models respond better to different prompt styles

3. **Validation is key**
   - Use callbacks/logging to inspect what's being sent
   - Verify the schema is actually in the messages

4. **Template customization helps**
   - Different models need different instruction styles
   - Create a registry of templates for your models

### Next Steps:

- Test with your actual vLLM endpoint
- Experiment with different template styles
- Monitor JSON output quality
- Consider contributing automatic injection to LiteLLM

### To Use With a Real vLLM Server:

```python
# Just remove the mocking and add your real API base
response = completion_with_schema(
    model="hosted_vllm/your-model",
    messages=messages,
    schema=your_schema,
    api_base="http://your-vllm-server:8000"
)
```